In [134]:
import pandas as pd
import numpy as np
import itertools
import random
from collections import Counter
from pathlib import Path
random.seed(1)


Single Finger Flexion and Extension to 4 different extents: +3, +1.5, -1.5, and -3 N. In each trial one target appears, all the transitions (16 transitions between these 4 conditions) should happen sudo randomly during two days. 

A random cursor perturbation with zero mean 0.5 N variance is added to determine the learning rates. 

In [135]:
iti = 1000 # msecs for inter-trial interval
planTime = 2000 # msecs for precue time
feedbackTime = 1000 # msecs for feedback time

purturbation_std = 0.5 # Newton

total_sub_num = 8
# num_trials_baseline = 20
num_trials_per_block = 32 # or 33 for the last block
num_trials_per_transition = 40

finger = 2  # index (1-based: thumb=1 ... pinky=5)
conditions = [-3.0, -1.5, 1.5, 3.0]


def generate_sequence(conditions, num_trials_per_transition, seed):
    """Eulerian circuit over the directed transition multigraph (Hierholzer)."""
    random.seed(seed)

    adj = {
        a: [b for b in conditions for _ in range(num_trials_per_transition)]
        for a in conditions
    }
    for dests in adj.values():
        random.shuffle(dests)

    stack = [random.choice(conditions)]
    sequence = []
    while stack:
        u = stack[-1]
        if adj[u]:
            stack.append(adj[u].pop())
        else:
            sequence.append(stack.pop())
    sequence.reverse()
    return sequence

    


In [157]:
seq = generate_sequence(conditions, num_trials_per_transition, 3)
transitions = [(seq[i], seq[i+1]) for i in range(len(seq) - 1)]
Counter(transitions)

Counter({(-4, -2): 40,
         (-2, -4): 40,
         (-4, 4): 40,
         (4, -4): 40,
         (4, -2): 40,
         (-2, 2): 40,
         (2, -2): 40,
         (-2, 4): 40,
         (-4, -4): 40,
         (-4, 2): 40,
         (4, 4): 40,
         (4, 2): 40,
         (2, 4): 40,
         (2, 2): 40,
         (2, -4): 40,
         (-2, -2): 40})

In [164]:
generate_sequence(conditions, num_trials_per_transition, 3)


[-4,
 -2,
 -4,
 4,
 -4,
 4,
 -2,
 2,
 -2,
 4,
 -4,
 -4,
 2,
 -2,
 4,
 4,
 4,
 4,
 2,
 4,
 4,
 -2,
 -4,
 4,
 -4,
 4,
 -2,
 -4,
 4,
 -2,
 2,
 4,
 -4,
 -4,
 -4,
 4,
 4,
 2,
 2,
 -4,
 -2,
 2,
 4,
 -2,
 2,
 2,
 -4,
 4,
 4,
 -2,
 4,
 2,
 -4,
 -2,
 -2,
 4,
 4,
 -2,
 4,
 -2,
 4,
 -4,
 -2,
 4,
 4,
 4,
 2,
 -4,
 4,
 -2,
 2,
 -4,
 4,
 -2,
 -2,
 2,
 -4,
 4,
 2,
 -4,
 4,
 -2,
 -2,
 2,
 -4,
 2,
 4,
 2,
 -4,
 -4,
 4,
 4,
 -4,
 4,
 -2,
 -2,
 2,
 -4,
 4,
 2,
 4,
 4,
 4,
 -2,
 -2,
 2,
 4,
 -4,
 2,
 4,
 4,
 -2,
 4,
 2,
 2,
 -2,
 -2,
 4,
 2,
 4,
 4,
 2,
 -4,
 4,
 -2,
 4,
 -4,
 4,
 4,
 4,
 -4,
 -2,
 -4,
 -4,
 -2,
 -4,
 -4,
 -2,
 -4,
 4,
 2,
 2,
 2,
 2,
 -2,
 -2,
 -2,
 2,
 2,
 -2,
 2,
 -2,
 4,
 4,
 2,
 2,
 -2,
 -4,
 -2,
 -2,
 -4,
 2,
 -4,
 2,
 -4,
 2,
 -2,
 2,
 4,
 2,
 2,
 4,
 4,
 4,
 -4,
 4,
 -4,
 2,
 4,
 4,
 2,
 2,
 -4,
 2,
 -2,
 -2,
 4,
 2,
 4,
 2,
 -4,
 4,
 -4,
 -2,
 2,
 -2,
 -4,
 -2,
 -4,
 2,
 -2,
 4,
 -4,
 2,
 4,
 -4,
 -2,
 4,
 2,
 2,
 2,
 -2,
 -2,
 4,
 4,
 -4,
 -2,
 2,
 2,
 2,
 2,
 2,
 -2,
 -4,
 -4,


In [131]:
# # check the number of transitions in sequence
# transitions = [(sequence[i], sequence[i+1]) for i in range(len(sequence) - 1)]
# Counter(transitions)

In [132]:
def chunk(xs, size):
    return [list(xs[i:i+size]) for i in range(0, len(xs), size)]

In [133]:
# target file columns match MyTrial::read:
# subNum, targetForce1-5, isTargetVisible, purturbation1, purturbation2, planTime, feedbackTime, iti
# index = targetForce2 / purturbation1; last block has 33 trials
# Each subject is independently seeded (seed = sub_num), but re-running still
# overwrites files. Skip subjects that already have target files so sn1–sn4 stay as-is.

output_dir = Path('TargetFiles')
output_dir.mkdir(exist_ok=True)

columns = ['subNum', 'targetForce1', 'targetForce2', 'targetForce3', 'targetForce4', 'targetForce5',
           'isTargetVisible', 'purturbation1', 'purturbation2', 'planTime', 'feedbackTime', 'iti']

for sub_num in range(1, total_sub_num + 1):
    if (output_dir / f'sn{sub_num}_1.tgt').exists():
        continue

    seed = sub_num
    np.random.seed(seed)
    sequence = generate_sequence(conditions, num_trials_per_transition, seed)
    noise = np.round(np.random.normal(0, purturbation_std, len(sequence)), 2)

    sequence_chunks = chunk(sequence[:-1], num_trials_per_block)
    noise_chunks = chunk(noise[:-1], num_trials_per_block)
    sequence_chunks[-1].append(sequence[-1])
    noise_chunks[-1].append(float(noise[-1]))

    bn = 1
    for sequence_chunk, noise_chunk in zip(sequence_chunks, noise_chunks):
        rows = []
        for target, pert in zip(sequence_chunk, noise_chunk):
            target_forces = [0.0] * 5
            target_forces[finger - 1] = float(target)
            rows.append([
                sub_num,
                *target_forces,
                1,
                float(pert),
                0.0,
                planTime,
                feedbackTime,
                iti,
            ])
        block_data = pd.DataFrame(rows, columns=columns).astype({
            'subNum': int,
            'isTargetVisible': int,
            'planTime': int,
            'feedbackTime': int,
            'iti': int,
        })
        block_data.to_csv(output_dir / f'sn{sub_num}_{bn}.tgt', index=False, sep='\t')
        bn += 1
